# Token Graph: Rearrangements × K-mers

Builds a **bipartite graph** from GILGFVFTL-specific TRB CDR3 sequences where:

* **Rearrangement nodes** — one per unique CDR3 sequence.
* **K-mer nodes** — one per unique annotated 3-mer.

An edge connects a rearrangement to every 3-mer appearing in its CDR3.

The notebook shows the full graph (one giant CC) **before** applying the RS filter,
highlighting which nodes will be removed, then shows the RS-filtered cluster.

In [1]:
import gzip
import time
from pathlib import Path

import igraph as ig
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import sys
sys.path.append("../")
from mir.basic.token_tables import (
    Rearrangement,
    filter_token_table,
    tokenize_rearrangements,
)
from mir.graph.token_graph import build_token_graph

## 1. Load data and build token tables

In [ ]:
GILG_FILE = Path("../tests/assets/gilgfvftl_trb_cdr3.txt.gz")
K = 3

with gzip.open(GILG_FILE, "rt", encoding="utf-8") as fh:
    cdr3s = [line.strip() for line in fh if line.strip()]

rearrangements = [
    Rearrangement(sequence_id=str(i), locus="TRB", v_gene="TRB", junction_aa=seq, duplicate_count=1)
    for i, seq in enumerate(cdr3s)
]

table    = tokenize_rearrangements(rearrangements, k=K)
rs_table = filter_token_table(table, kmer_pattern="RS")

rs_kmer_seqs = {k.seq.decode() for k in rs_table}

print(f"Sequences      : {len(rearrangements)}  ({sum('RS' in s for s in cdr3s)} with RS)")
print(f"Total {K}-mers  : {len(table)}")
print(f"RS {K}-mers     : {len(rs_table)}  {sorted(rs_kmer_seqs)}")

## 2. Full graph — one connected component

All CDR3s share common non-RS k-mers (e.g. `CAS`, `ASS`), so the full
token graph forms a **single giant CC**.  After RS filtering, non-RS rearrangements
become isolated singletons and non-RS k-mers are removed from the graph.

In [ ]:
g_full = build_token_graph(rearrangements, table)
g_rs   = build_token_graph(rearrangements, rs_table)

full_cc = g_full.components().giant()
filt_cc = g_rs.components().giant()

# Sets of junction_aa in filtered CC (to label 'retained' nodes in full CC)
retained_r = {v["name"] for v in filt_cc.vs if v["node_type"] == "rearrangement"}
retained_k = {v["name"] for v in filt_cc.vs if v["node_type"] == "kmer"}

n_full_r     = sum(1 for v in full_cc.vs if v["node_type"] == "rearrangement")
n_full_k     = sum(1 for v in full_cc.vs if v["node_type"] == "kmer")
n_nonrs_r    = sum(1 for v in full_cc.vs if v["node_type"] == "rearrangement" and v["name"] not in retained_r)
n_nonrs_k    = sum(1 for v in full_cc.vs if v["node_type"] == "kmer" and v["name"] not in retained_k)
n_rs_r       = len(retained_r)
n_rs_k       = len(retained_k)

print(f"Full CC : {full_cc.vcount()} nodes  ({n_full_r} rearrangements, {n_full_k} k-mers)")
print(f"  → will be REMOVED : {n_nonrs_r} rearrangements + {n_nonrs_k} k-mers")
print(f"  → will be KEPT    : {n_rs_r} rearrangements + {n_rs_k} k-mers")
print(f"RS-filtered giant CC: {filt_cc.vcount()} nodes  ({n_rs_r} rearrangements, {n_rs_k} k-mers)")

## 3. Visualise the full CC — what gets filtered

Node colours in the **full graph** (before filtering):

| Colour | Meaning |
|--------|---------|
| Red    | Non-RS k-mer — **will be removed** |
| Salmon | Non-RS rearrangement — **will be isolated** |
| Orange | RS k-mer — **retained** (labelled) |
| Blue   | RS rearrangement — **retained** |

> A random layout is used (instant).  Spatial position
> is not meaningful here — the colours reveal what gets filtered.

In [ ]:
subg = full_cc
degrees = subg.degree()

# Bipartite layout: rearrangements on one row, k-mers on the other.
# igraph requires a boolean "type" attribute (True = upper partition).
subg.vs["type"] = [v["node_type"] == "kmer" for v in subg.vs]

vertex_colors = []
vertex_sizes  = []
vertex_labels = []

for v in subg.vs:
    is_kmer = v["node_type"] == "kmer"
    is_rs   = v["name"] in (retained_k if is_kmer else retained_r)
    if is_kmer:
        if is_rs:
            vertex_colors.append("#e67e22")   # orange — RS kmer, retained
            vertex_sizes.append(14 + degrees[v.index] // 20)
            vertex_labels.append(v["name"])
        else:
            vertex_colors.append("#e74c3c")   # red — non-RS kmer, will be removed
            vertex_sizes.append(3)
            vertex_labels.append("")
    else:  # rearrangement
        if is_rs:
            vertex_colors.append("#2980b9")   # blue — RS rearrangement, retained
            vertex_sizes.append(4)
        else:
            vertex_colors.append("#f1948a")   # salmon — non-RS rearrangement, will be isolated
            vertex_sizes.append(2)
        vertex_labels.append("")

layout = subg.layout_bipartite()   # two-row bipartite layout; position encodes node type

fig, ax = plt.subplots(figsize=(18, 6))
ig.plot(
    subg, target=ax, layout=layout,
    vertex_color=vertex_colors,
    vertex_size=vertex_sizes,
    vertex_label=vertex_labels,
    vertex_label_size=7,
    vertex_label_color="black",
    edge_width=0.05,
    edge_color="#e8e8e8",
)

legend_handles = [
    mpatches.Patch(color="#e74c3c", label=f"Non-RS k-mer — removed ({n_nonrs_k})"),
    mpatches.Patch(color="#f1948a", label=f"Non-RS rearrangement — isolated ({n_nonrs_r})"),
    mpatches.Patch(color="#e67e22", label=f"RS k-mer — kept, labelled ({len(retained_k)})"),
    mpatches.Patch(color="#2980b9", label=f"RS rearrangement — kept ({len(retained_r)})"),
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=8)
ax.set_title(
    f"Full token graph — largest CC ({subg.vcount()} nodes), bipartite layout\n"
    f"Top row: k-mers ({n_full_k})   Bottom row: rearrangements ({n_full_r})\n"
    f"Colour shows what RS-filtering will keep vs. remove",
    fontsize=10,
)
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RS-filtered graph — retained cluster

After filtering, only RS-bearing rearrangements connected via shared RS k-mers
remain in the giant CC.  All non-RS rearrangements become isolated singletons.

In [ ]:
subg_rs = filt_cc
degrees_rs = subg_rs.degree()

rs_colors = [
    "#e67e22" if v["node_type"] == "kmer" else "#2980b9"
    for v in subg_rs.vs
]
rs_sizes = [
    16 + degrees_rs[v.index] if v["node_type"] == "kmer" else 4
    for v in subg_rs.vs
]
rs_labels = [
    v["name"] if v["node_type"] == "kmer" else ""
    for v in subg_rs.vs
]

layout_rs = subg_rs.layout("fr")

n_r_cc = sum(1 for v in subg_rs.vs if v["node_type"] == "rearrangement")
n_k_cc = sum(1 for v in subg_rs.vs if v["node_type"] == "kmer")
n_isolated = len(rearrangements) - n_rs_r

fig, ax = plt.subplots(figsize=(14, 12))
ig.plot(
    subg_rs, target=ax, layout=layout_rs,
    vertex_color=rs_colors,
    vertex_size=rs_sizes,
    vertex_label=rs_labels,
    vertex_label_size=7,
    vertex_label_color="black",
    edge_width=0.4,
    edge_color="#bdc3c7",
)

legend_handles = [
    mpatches.Patch(color="#e67e22", label=f"RS k-mer ({n_k_cc}) — labelled"),
    mpatches.Patch(color="#2980b9", label=f"RS rearrangement ({n_r_cc})"),
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=9)
ax.set_title(
    f"RS-filtered token graph — largest CC ({subg_rs.vcount()} nodes)\n"
    f"{n_r_cc} rearrangements + {n_k_cc} RS k-mers  "
    f"({n_isolated} non-RS rearrangements are now isolated)",
    fontsize=11,
)
ax.axis("off")
plt.tight_layout()
plt.show()